In [ ]:
!pip install camel_tools

In [ ]:
from camel_tools.ner import NERecognizer
from camel_tools.tokenizers.word import simple_word_tokenize

import re


In [ ]:

def has_no_arabic_chars(word):

    pattern =  r'[«#>\]_*,;%@+/:^)~»<?$؛({\'!}=.\[\\"`،|\-&]'
    return re.sub(pattern, '', word)


def clean_data(word, label):
  if has_no_arabic_chars(word) =='': return [], []
  else : return has_no_arabic_chars(word), label

def split_text_file(filename):
    with open(filename ,'r', encoding='utf-8') as file:
        lines = file.readlines()
    data = [];
    for line in lines:
        line = line.strip()
        if line:
            parts = line.split('\t')
            text = parts[0]
            _sentence = text.split(" ")
            _labels = parts[1:][0].split(" ")
            sentence= []; labels =[]
            for i, word in enumerate(_sentence):
              tmp_sen, tmp_leb = clean_data(word, _labels[i])
              if tmp_sen != []: sentence.append(tmp_sen); labels.append(tmp_leb)
            data.append((sentence, labels))

    return data


data= split_text_file("/content/drive/MyDrive/projects/NER/b.txt")

In [ ]:

ner = NERecognizer('CAMeL-Lab/bert-base-arabic-camelbert-mix-ner')

Some weights of the model checkpoint at CAMeL-Lab/bert-base-arabic-camelbert-mix-ner were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:

label_convert = {
    "B-LOC":"loc", "I-LOC":"loc","B-ORG":"org","I-ORG":"org",
    "I-PERS":"per", "B-PERS":"per", "O":"O", "B-COM": "O",
    "I-COM": "O","B-MISC" :"misc", "I-MISC":"misc",
}


def predict_data(sentence, nlp):
  annotations = nlp.predict_sentence(sentence)
  entities = []
  tags = []
  for i, _sentence in enumerate(sentence):
    entities.append(_sentence) ; tags.append(label_convert[annotations[i]])
  return entities, tags

predicted_data = []

data_values = []
count = 0
print(len(data))
for X, y in data:
  if count%50 == 0: print(count); print(X)
  data_values.append((X, y))
  predicted_data.append(predict_data(X, ner))
  count +=1



922
0
['الصالحية', 'المفرق', 'غيث', 'الطراونة', 'أمر', 'جلالة', 'الملك', 'عبدالله', 'الثاني', 'أمس', 'بتنفيذ', 'حزمة', 'من', 'المشاريع', 'التعليمية', 'والصحية', 'والتنموية', 'وأخرى', 'مرتبطة', 'بالأندية', 'الشبابية', 'و', '27', 'وحدة', 'سكنية', 'في', 'قضاء', 'الصالحية', 'ونايفة', 'في', 'البادية', 'الشرقية', 'خلال', 'ستة', 'اشهر', 'بتمويل', 'من', 'الديوان', 'الملكي', 'الهاشمي']
50
['بلدة', 'إن', 'وودورد', 'رسم', 'في', 'كتابه', 'حالة', 'النكران', 'صورة', 'للصراعات', 'الشخصية', 'الحادة', 'داخل', 'الإدارة', 'الأميركية', 'مشيرا', 'إلى', 'أن', 'آندرو', 'كارل', 'رئيس', 'موظفي', 'إدارة', 'البيت', 'الأبيض', 'السابق', 'حاول', 'مرتين', 'إجبار', 'دونالد', 'رمسفيلد', 'وزير', 'الدفاع', 'الأميركي', 'على', 'الاستقالة', 'على', 'خلفية', 'إدارته', 'للحرب', 'على', 'العراق']
100
['ثانية', 'البابا', 'قد', 'اقتبس', 'في', 'محاضرة', 'ألقاها', 'في', 'راتيسبون', 'بألمانيا', 'في', '12', 'سبتمبر', 'أيلول', 'عن', 'إمبراطور', 'بيزنطي', 'في', 'القرن', 'ال', 'ـ', '14', 'حديثا', 'عن', 'العلاقة', 'بين', 'الإسلام', 'والع

In [ ]:
print(len(predicted_data[0][1] ),'\n', len(data_values[0][1]) ,'\n', len(data_values[0][0]))

40 
 40 
 40


In [ ]:
grouped_data = []
count = 0
for words, true_label in data_values:
    pred_words, pred_label = predicted_data[count]
    count +=1
    for i, word in enumerate(words):
      if word == pred_words[i] :
        grouped_data.append([word, true_label[i], pred_label[i]])
      else:
        grouped_data.append([word, true_label[i], pred_label[i]])

In [ ]:
import pandas as pd



df = pd.DataFrame(grouped_data, columns=['Word', 'Target', 'Predicted'])
df['Target'] = df['Target'].map({
    "B-PERS":"per", "I-PERS"	:"per", "O":"O",
    "B-LOC": "loc",  "B-evnt": "evnt","B-ORG": "org","B-MISC": "misc",
    "I-LOC": "loc",  "I-evnt": "evnt","I-ORG": "org","I-MISC": "misc",
    })

df

,Word,Target,Predicted
0,الصالحية,loc,loc
1,المفرق,loc,loc
2,غيث,per,per
3,الطراونة,per,per
4,أمر,O,O
...,...,...,...
22504,الوزيرة,O,O
22505,ابن,O,O
22506,وابنة,O,O
22507,خالد,per,per


In [ ]:
df = df.dropna()

In [ ]:
from sklearn.metrics import classification_report
report = classification_report(list(df['Target']), list(df['Predicted']))
print(report)

              precision    recall  f1-score   support

           O       0.98      0.99      0.99     19138
         loc       0.88      0.94      0.91       751
        misc       0.81      0.50      0.62       398
         org       0.82      0.74      0.78       725
         per       0.93      0.93      0.93      1496

    accuracy                           0.97     22508
   macro avg       0.89      0.82      0.84     22508
weighted avg       0.97      0.97      0.97     22508



In [ ]:
from sklearn.metrics import confusion_matrix
conf_matrix = confusion_matrix(df['Target'], df['Predicted'], labels=['loc', 'misc', 'org', 'per', 'O'])

In [ ]:
conf_matrix

array([[  703,     1,    10,     5,    32],
       [   24,   198,    31,    11,   134],
       [   34,     6,   534,    35,   116],
       [   14,    12,    21,  1384,    65],
       [   20,    28,    53,    49, 18988]])

In [ ]:
digits_convert ={"loc": 1,  "misc": 2,"org": 3, "per": 4, "O": 5,}
df['Target'] = df['Target'].map(digits_convert)
df['Predicted'] = df['Predicted'].map(digits_convert)
df.head()

<ipython-input-131-5d7feae604e2>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Target'] = df['Target'].map(digits_convert)
<ipython-input-131-5d7feae604e2>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Predicted'] = df['Predicted'].map(digits_convert)


,Word,Target,Predicted
0,الصالحية,1,1
1,المفرق,1,1
2,غيث,4,4
3,الطراونة,4,4
4,أمر,5,5


In [ ]:
from sklearn.metrics import f1_score
labels_ = {}
f1_score(df['Target'], df['Predicted'], average='macro')

0.8437080707397353

In [ ]:
f1_score(df['Target'], df['Predicted'], average='micro')


0.9688555180380309

In [ ]:
f1_score(df['Target'], df['Predicted'], average='weighted')

0.9673000034849879

In [ ]:
f1_scores =f1_score(df['Target'], df['Predicted'], average=None)

In [ ]:

results = pd.DataFrame(conf_matrix, columns=['loc', 'misc', 'org', 'per', 'O'])
results['f1_score'] = f1_scores
results

,loc,misc,org,per,O,f1_score
0,703,1,10,5,32,0.909444
1,24,198,31,11,134,0.615863
2,34,6,534,35,116,0.777293
3,14,12,21,1384,65,0.928859
4,20,28,53,49,18988,0.987082
